# LOG05 | 미션 02 | Student Workbook

## 이번 미션에서 하는 일

**투입 전 점검(배터리/유류, 통신 링크, 적재 중량, 기상) 체크리스트 작성**

1. 실습 목적 및 개요 역할 및 배경: 학습자는 무인 보급장비 투입 전 상태·책임·중지조건 체크리스트 교육용 실습의 참여자로서 임무를 수행합니다. 핵심 과제: 투입 전 점검 항목(배터리/유류, 통신 링크, 적재 중량, 기상)을 합격 기준과 함께 체크리스트로 만들고, 점검 결과를 얼마나 믿을 수 있는지 6단계(G1～G6)로 구분합니다. 2. 주의 및 검증 지침 데이터 해석의 엄정성: 자료나 AI 설명을 철저히 검증하지 않으면 미확인 기록이나 모의 회신을 실제 현장 확인이나 승인으로 오인할 수 있으므로 주의해야 합니다. 수행 방식: 본인 판단, AI 초안 및 역할 대화, 근거 대조 및 보완, 최종 기록의 전 과정을 원본 자료 범위 내에서 수행합니다. 3. 산출물 및 소요 시간 안내 결과 파일 목록: framework/log05_source_reliability_matrix.md LOG05_M02_forms.md forms/LOG05_M02_forms.md 시간 안내: 원본 예상 시간은 2시간이며, CMS 편성은 원본 durationHours 를 반영하여 2시간으로 구성되어 있습니다 (실제 소요 시간은 실습 완료 후 확인).

**학습 목표:** 가상 자료를 읽고 자신의 최초 판단을 적은 다음, AI 초안을 원자료와 대조하여 문서와 검증 근거를 완성합니다. 실제 장비 제어·비행·파괴·무력화 방법을 설계하거나 실행하는 훈련은 아닙니다.

**준비물:** 이 Notebook, 미션 2의 가상 자료 및 빈 작성 양식, 기관이 허용한 AI 환경(가능한 경우).

**학습 환경:** Colab은 인터넷이 필요합니다. 폐쇄망에서는 사전 설치된 Jupyter 또는 별도 Markdown 양식으로 진행할 수 있으며 외부 AI를 사용할 수 없으면 미실시로 기록하세요.

**학습 순서:** 자료 먼저 읽기 → 본인 생각 기록 → AI 질문 → 원문 근거 대조 → 수정 → 저장·제출.

**구분:** 파일 저장 / 실습 수행 / 평가 기준 충족 / 실제 교관 승인은 각각 다른 상태입니다.

## 📦 실습자료 업로드 (모든 미션 공통 · 실습 전 필수)

아래 **「LOG05 실습자료 ZIP 업로드 및 압축 해제」** 코드셀을 **먼저 실행**하세요.

1. 교육기관에서 제공한 **`LOG05_실습프로젝트.zip`**을 준비합니다. 파일명이 다르더라도 LOG05 자료가 담긴 정상적인 ZIP 파일이면 업로드할 수 있습니다.
2. **Google Colab:** 셀 실행 후 나타나는 업로드 창에서 ZIP을 선택하면 자동으로 압축을 해제합니다. **로컬 Jupyter:** ZIP 선택 후 **「선택한 ZIP 압축 해제」** 버튼을 누릅니다.
3. 화면에서 **「압축 해제 완료」** 메시지와 실습자료 폴더 경로를 확인한 다음, 아래 미션 실습을 진행하세요.

**주의:** 업로드가 끝나지 않았다면 미션 실습을 시작하지 마세요. ZIP을 올렸다는 사실만으로 실습 결과나 교관 승인이 완료되는 것은 아닙니다. 아래 코드셀의 소스코드는 기본 숨김으로 설정되어 있습니다.


In [ ]:
# @title 📦 LOG05 실습자료 ZIP 업로드 및 압축 해제 { display-mode: "form" }
# 학습자 화면에는 결과와 업로드 인터페이스만 표시하도록 구성합니다.
from pathlib import Path
import os, zipfile, stat
from IPython.display import display, Markdown

LOG05_PROJECT_DIR = Path.cwd() / "LOG05_실습자료"
LOG05_PROJECT_READY = False

def _log05_extract_zip(zip_path):
    global LOG05_PROJECT_READY
    zip_path = Path(zip_path)
    if not zipfile.is_zipfile(zip_path):
        raise ValueError("유효한 ZIP 압축파일이 아닙니다. 실습자료 ZIP을 다시 선택해 주세요.")
    dst = LOG05_PROJECT_DIR.resolve()
    dst.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        entries = archive.infolist()
        if len(entries) > 20000 or sum(info.file_size for info in entries) > 1024**3:
            raise ValueError("압축파일의 항목 수 또는 해제 후 용량이 허용 범위를 초과합니다.")
        for info in entries:
            item = Path(info.filename.replace('\\', '/'))
            if item.is_absolute() or '..' in item.parts or not info.filename.strip():
                raise ValueError("안전하지 않은 ZIP 내부 경로가 포함되어 있습니다: " + info.filename)
            target = (dst / item).resolve()
            if not target.is_relative_to(dst):
                raise ValueError("실습 폴더 밖으로 벗어나는 파일 경로입니다.")
            mode = (info.external_attr >> 16) & 0o170000
            if mode == stat.S_IFLNK:
                raise ValueError("심볼릭 링크가 포함된 ZIP은 허용하지 않습니다.")
        for info in entries:
            target = dst / Path(info.filename.replace('\\', '/'))
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as src, target.open('wb') as out:
                    import shutil
                    shutil.copyfileobj(src, out)
    files = [p for p in dst.rglob('*') if p.is_file()]
    if not files:
        raise ValueError("ZIP에 압축 해제할 실습자료 파일이 없습니다.")
    LOG05_PROJECT_READY = True
    print("✅ LOG05 실습자료 압축 해제 완료:", dst)
    print("📄 확인된 파일 수:", len(files))
    print("ℹ️ 실습자료 폴더:", dst)
    print("※ 업로드·압축 해제는 학습 수행 및 평가 완료를 뜻하지 않습니다.")
    return dst

display(Markdown("**실습 시작 전 필수:** 교육기관에서 제공한 `LOG05_실습프로젝트.zip`을 업로드하세요. 이름이 다르더라도 정상적인 `.zip` 파일을 선택할 수 있지만, **LOG05 실습자료가 포함된 ZIP**이어야 합니다. 업로드 및 압축 해제를 완료한 후 다음 실습 단계로 진행하세요."))
try:
    from google.colab import files as _colab_files
    print("📤 업로드 창에서 LOG05 실습자료 ZIP 파일을 선택하세요.")
    _uploaded = _colab_files.upload()
    _zip_files = [(name, content) for name, content in _uploaded.items() if name.lower().endswith('.zip')]
    if not _zip_files:
        print("⚠️ ZIP 파일이 업로드되지 않았습니다. 이 셀을 다시 실행하세요.")
    else:
        if len(_zip_files) > 1:
            print("ℹ️ 여러 ZIP 중 첫 번째 파일을 사용합니다:", _zip_files[0][0])
        _name, _data = _zip_files[0]
        _zip_path = Path.cwd() / Path(_name).name
        _zip_path.write_bytes(_data)
        _log05_extract_zip(_zip_path)
except ImportError:
    import ipywidgets as _widgets
    _file_input = _widgets.FileUpload(accept='.zip', multiple=False, description='ZIP 파일 선택')
    _extract_button = _widgets.Button(description='선택한 ZIP 압축 해제', button_style='success')
    _upload_message = _widgets.Output()
    def _get_uploaded_zip():
        value = _file_input.value
        if not value:
            raise ValueError('먼저 LOG05 실습자료 ZIP을 선택하세요.')
        item = next(iter(value.values())) if isinstance(value, dict) else value[0]
        name = item['name']
        data = item['content']
        if not name.lower().endswith('.zip'):
            raise ValueError('ZIP 파일만 선택할 수 있습니다.')
        return name, bytes(data)
    def _on_extract(_):
        with _upload_message:
            _upload_message.clear_output()
            try:
                name, data = _get_uploaded_zip()
                target = Path.cwd() / Path(name).name
                target.write_bytes(data)
                _log05_extract_zip(target)
            except Exception as exc:
                print('⚠️ 업로드 또는 압축 해제 실패:', exc)
    _extract_button.on_click(_on_extract)
    display(_widgets.VBox([_file_input, _extract_button, _upload_message]))
    print('로컬 Jupyter: ZIP을 선택한 뒤 「선택한 ZIP 압축 해제」 버튼을 누르세요.')


## 원본 가상 자료와 작성 양식

아래는 첨부된 **LOG05 실습자료 원본**에서 미션별로 추출한 자료입니다. 모의 기록에서 아직 공개되지 않은 카드나 결과는 미리 확정하지 마세요.

미션 2. 투입 전 점검(배터리/유류, 통신 링크, 적재 중량, 기상) 체크리스트 작성

**가. 제공 자료 1 - 가상 점검 보고 5종 [교육용가상자료]**

| 보고 | 장비·영역 | 보고 내용 | 확인 방법 | 확인 상태 |
| --- | --- | --- | --- | --- |
| R1 | UD-1 배터리 | 잔량 82%, 부하 시험 전압 정상 | 계측기 값 기록 + 정비 담당 재확인 서명 | 계측값과 서명 모두 있음 |
| R2 | UV-2 링크 | 품질 4등급 | 시험 송수신 1회 로그 | 로그 있으나 1회뿐 |
| R3 | UD-1 적재 | 적재 7단위 | 적재 담당 자기 신고 | 저울 기록 없음 |
| R4 | C 구간 기상 | 괜찮음 | 근거 없음 | 풍속·시정 값 없음 |
| R5 | UV-2 경로 | 최신 상태 | 전날 점검표 인용 | 하루 전 자료, 변경 여부 미확인 |

**가. 제공 자료 2 - 점검 결과 확인 신뢰도 6단계 틀. 단계 이름은 고정하고 판정 기준을 조별로 작성한다.**

| 단계 | 이름 | 판정 기준(조별 작성) |
| --- | --- | --- |
| G1 | 계측값·이중 확인 |  |
| G2 | 계측값 기록 |  |
| G3 | 시험·로그 확인 |  |
| G4 | 전일 자료·인용 |  |
| G5 | 자기 신고·구두 |  |
| G6 | 근거 없음 |  |

**나. 작성 서식 1 - 투입 전 점검 항목표(10개 이상, 4개 영역 모두 포함, 예시 2개 포함)**

| 번호 | 영역 | 점검 항목 | 합격 기준 | 확인 방법 | 확인자(가상) | 불합격 시 |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | 배터리 | 부하 시험 후 잔량 | 출발 시 90% 이상, 예비 전력 포함 왕복 가능 | 계측기 기록 + 이중 확인 | 정비 담당 | 보류, 충전 후 재점검 |
| 2 | 통신 | 링크 품질 등급 | 3등급 이상 + 시험 송수신 3회 성공 | 통신 로그 | 통신 담당 | 보류, 경로 변경 검토 |
| 3 |  |  |  |  |  |  |
| 4 |  |  |  |  |  |  |
| 5 |  |  |  |  |  |  |
| 6 |  |  |  |  |  |  |
| 7 |  |  |  |  |  |  |
| 8 |  |  |  |  |  |  |
| 9 |  |  |  |  |  |  |
| 10 |  |  |  |  |  |  |
| 11 |  |  |  |  |  |  |
| 12 |  |  |  |  |  |  |

**나. 작성 서식 2 - 등급 부여 일관성 시험(조원이 각자 G1～G6을 매기고 비교)**

| 보고 | 조원 1 | 조원 2 | 조원 3 | 조원 4 | 일치 여부 | 불일치 원인 |
| --- | --- | --- | --- | --- | --- | --- |
| R1 |  |  |  |  |  |  |
| R2 |  |  |  |  |  |  |
| R3 |  |  |  |  |  |  |
| R4 |  |  |  |  |  |  |
| R5 |  |  |  |  |  |  |

**나. 작성 서식 3 - 교차 확인 필수 규칙: 점검자 자기 신고만으로는 G (   ) 이하로 기록하고 투입 승인은 서로 다른 확인자 (   )명 이상의 확인이 있을 때만 검토한다. 점검 후 (   )분이 지난 결과는 (   )로 처리하고 재점검한다.**

**다. 점검표**

| 점검 항목 | 확인 |
| --- | --- |
| 4개 점검 영역(배터리/유류, 통신, 적재, 기상)을 모두 포함했다 | (   ) |
| 점검 항목을 10개 이상 정의하고 합격 기준을 수치 또는 관찰 가능한 문장으로 적었다 | (   ) |
| 신뢰도 G1～G6의 판정 기준을 모두 작성했다 | (   ) |
| 가상 점검 보고 5종에 대한 조원 등급이 4건 이상 일치했다(불일치 건은 원인과 수정 내용을 적었다) | (   ) |
| 자기 신고 단독 의존 경고와 교차 확인 필수 규칙을 명시했다 | (   ) |

## 단계별 수행 안내와 미션 전용 AI 활용

### 활동 1. 근거 등급과 합격을 분리한다

**무엇을 하나요?** 1. 단계 개요 목표: 가상 점검 보고 5종( R1 ～ R5 )을 직접 분석하여 각각의 확인 방법과 상태를 검토하고, 점검 결과의 신뢰도 등급( G1 ～ G6 )과 실제 합격/불합격 여부를 명확히 분리하여 판정 기준의 타당성을 검증합니다. 수행 방식: 단편적인 합격 여부 판단에 그치지 않고, 해당 점검 결과가 어떤 근거와 신뢰 수준(계측값, 로그, 자기 신고 등)에 기반했는지를 엄밀히 대조하여 주관적 판단의 오류를 방지합니다. 2. 세부 수행 절차 가상 점검 보고 5종(R1～R5) 검토: UD-1 배터리( R1 ), UV-2 링크( R2 ), UD-1 적재( R3 ), C 구간 기상( R4 ), UV-2 경로( R5 ) 보고 내용을 각각 분석하고 실제 확인 방법 및 확인 상태를 확인합니다. 근거 등급(G1～G6)과 합격 여부의 분리 판정: 보고별로 단순히 '합격' 또는 '불합격'을 매기는 데 그치지 않고, 해당 결과가 계측기 기록과 이중 확인( G1 ), 단순 계측값 기록( G2 ), 시험 및 로그 확인( G3 ), 전일 자료 인용( G4 ), 자기 신고 및 구두 보고( G5 ), 근거 없음( G6 ) 중 어디에 해당하는지 신뢰도 등급을 각각 부여합니다. 기준 충족·미충족 및 추가 확인 사항 표시: 각 보고 내용이 투입 전 점검 기준을 충족하는지, 아니면 근거 미비나 단일 출처 의존으로 인해 추가 확인이 필요한지 여부를 명시적으로 표시합니다. 3. 주의 및 검증 사항 점검자 한 사람의 자기 신고나 구두 보고만으로 이뤄진 결과를 충분한 검증 없이 곧바로 승인 처리하지 않도록 주의해야 합니다. 실제 수행한 분류 및 검토 결과는 비어 있는 상태로 두지 않고, 실습 기록 서식( forms/LOG05_M0

**직접 물어볼 AI 프롬프트:** 제공 기록 초안·근거 대조·검산·레드팀·모의 역할 대화

**검토:** 원자료 번호를 표시하고 실제 AI 회신을 수정한 이유를 적으세요.

### 활동 2. AI와 체크리스트·G 기준을 작성한다

**무엇을 하나요?** 1. 단계 개요 목표: 앞서 검토한 보고 내용을 바탕으로 배터리/유류, 통신 링크, 적재 중량, 기상 등 4개 영역을 모두 포함하는 10개 이상의 투입 전 점검 항목표 를 설계하고, 점검 결과의 신뢰도를 판정할 6단계 기준(G1～G6)과 교차 확인 필수 규칙 을 정립합니다. 수행 방식: 단편적이거나 주관적인 점검에 의존하지 않고, 관찰 가능한 수치나 명확한 문장으로 합격 기준과 불합격 시 조치 사항을 구체화합니다. 2. 세부 수행 절차 4개 영역 점검 항목 10개 이상 설계: 배터리/유류, 통신 링크, 적재 중량, 기상 4개 영역이 누락되지 않도록 배분하고, 각 항목마다 명확한 합격 기준(수치 또는 관찰 가능한 문장), 확인 방법, 확인자 직책, 불합격 시 조치 사항을 정의하여 10개 이상의 항목표를 완성합니다. 신뢰도 6단계(G1～G6) 판정 기준 작성: 계측값 및 이중 확인( G1 ), 계측값 기록( G2 ), 시험 및 로그 확인( G3 ), 전일 자료 인용( G4 ), 자기 신고 및 구두 보고( G5 ), 근거 없음( G6 )의 단계별 판정 기준을 구체적으로 확립합니다. 교차 확인 필수 규칙 및 단일 출처 통제 반영: 점검자 개인의 자기 신고만으로는 투입을 승인할 수 없도록 제한하고, 서로 다른 확인자 복수의 확인이 있을 때만 검토하도록 하는 교차 확인 규칙과 일정 시간이 지난 점검 결과의 처리 방안을 명시합니다. 3. 주의 및 검증 사항 점검 항목의 합격 기준이 모호하거나 주관적인 표현("상태 양호", "적당함" 등)으로 작성되지 않았는지 점검하고, 수치나 관찰 가능한 명확한 조건으로 기술되었는지 확인합니다. 작성된 체크리스트와 신뢰도 기준은 다음 미션(M3～M5)에서의 시나리오 실습 및 

**직접 물어볼 AI 프롬프트:** 제공 기록 초안·근거 대조·검산·레드팀·모의 역할 대화

**검토:** 원자료 번호를 표시하고 실제 AI 회신을 수정한 이유를 적으세요.

### 활동 3. 고정 기준으로 독립 검토를 받는다

**무엇을 하나요?** 1. 단계 개요 목표: 앞서 작성한 점검 항목과 신뢰도 등급(G1～G6) 기준을 외부에 공개하거나 최종 확정하기 전에, 실제 조원 또는 분리된 AI 모의 등급 검토를 통해 5건의 점검 보고 평가 일치율 을 직접 계산하고 기준의 타당성을 검증·보완합니다. 수행 방식: 주관적 판단이나 단일 시각에 갇히지 않고, 객관적인 고정 평가 기준을 바탕으로 타인(또는 분리된 AI)의 독립된 검토 의견을 수렴하여 판정의 신뢰도를 높입니다. 2. 세부 수행 절차 독립 검토 수행 (실제 조원 또는 분리된 AI 활용): 작성된 6단계 신뢰도 기준과 체크리스트를 활용하여, 평가에 참여하지 않은 조원이나 별도로 분리된 AI 세션을 통해 가상 점검 보고 5건( R1 ～ R5 )에 대한 모의 등급 판정을 진행합니다. 일치율 계산 및 편차 분석: 본인의 판정 결과와 독립 검토자의 판정 결과 간의 일치 건수를 확인하고, 5건 중 일치율(%)을 산출합니다. 의견이 엇갈린 항목이 있다면 어느 기준이 모호했는지 그 원인을 분석합니다. 평가 기준 보완 및 수정: 일치율 분석 과정에서 도출된 모호한 문구나 주관적 해석의 여지가 있는 판정 기준을 명확하게 수정·보완하여 평가의 객관성을 확보합니다. 3. 주의 및 검증 사항 형식적인 검토에 그치지 않고, 실제 일치율 계산 과정과 수정된 내용의 근거를 기록으로 남겨야 합니다. 실제 독립 검토나 일치율 산출 과정을 거치지 않은 채 임의로 완료 상태로 처리하지 않도록 주의합니다.

**직접 물어볼 AI 프롬프트:** 제공 기록 초안·근거 대조·검산·레드팀·모의 역할 대화

**검토:** 원자료 번호를 표시하고 실제 AI 회신을 수정한 이유를 적으세요.

### 활동 4. 최초·수정 기준과 결과를 저장한다

**무엇을 하나요?** 1. 단계 개요 목표: 앞선 단계에서 수행한 가상 점검 보고 분석 결과, 6단계 신뢰도 판정 기준(G1～G6), 그리고 독립 검토(실제 조원 또는 모의 AI)를 거치면서 수정·보완된 내용들을 체계적으로 정리하여 후속 미션(M3～M5)으로 원활히 인계합니다. 수행 방식: 단발성 실습 기록에 그치지 않고, 최초 수립한 기준과 검토 과정에서 변경된 수정 사항을 명확히 구분하여 기록함으로써 향후 시나리오 실습 및 최종 체크리스트 통합 과정의 추적성을 확보합니다. 2. 세부 수행 절차 최초 기준과 수정된 기준의 비교 정리: 최초 작성했던 점검 항목 및 신뢰도 판정 기준(G1～G6)과, 독립 검토 및 일치율 분석 과정을 거쳐 보완·확정된 최종 기준의 변경 이력을 명확히 정리합니다. 실제 검토 및 모의 검토 결과 반영: 조원 간 실제 토의 결과 혹은 분리된 AI 모의 등급 검토 결과를 바탕으로 산출된 일치율과, 의견이 엇갈렸던 항목에 대한 사후 조치 내용을 기록합니다. 미실시 항목 및 예외 조건(4/5요건 등) 명시: 현장 여건이나 데이터 부족 등으로 인해 충분히 검증되지 못한 부분이나 미실시된 항목이 있다면 이를 솔직하게 남겨, 후속 미션(M3 장비 중지 조건 정의, M5 시나리오 실습 등)에서 추가로 보완할 수 있도록 근거를 전달합니다. 최종 산출물 파일 연계: 정리된 내용을 framework/log05_source_reliability_matrix.md , LOG05_M02_forms.md , forms/LOG05_M02_forms.md 등 지정된 결과 파일 경로에 빠짐없이 반영하고 저장합니다. 3. 주의 및 검증 사항 실제 검토나 수정 과정을 거치지 않았음에도 임의로 모든 항목이 완벽하게 완료된 것처럼

**직접 물어볼 AI 프롬프트:** 제공 기록 초안·근거 대조·검산·레드팀·모의 역할 대화

**검토:** 원자료 번호를 표시하고 실제 AI 회신을 수정한 이유를 적으세요.

## 제출 전에 확인할 기준

**평가 항목 1:** [정량적 평가] [필수] 점검 체크리스트·신뢰도 6단계 — 신뢰도 평가 기준 6단계를 완비하고 평가 지표(점검 항목) 10개 이상을 정의했는가, 4개 점검 영역이 모두 포함되었는가, 가상 점검 보고 5종에 대한 조원 간 등급 일치가 4건 이상이며 단일 출처(자기 신고) 의존 경고와 교차 확인 필수 규칙을 명시했는가. PASS 증거 제공된조건·단위·기간·분모와실제수치/건수/항목근거를연결합니다. 목표미달·0기준·미측정은그대로표시합니다. 보완 방법 기준별빠진항목과미달원인을찾아수정하고재검증합니다.

**평가 항목 2:** [정성적 평가] Why/How/What: 본인 계획·실제 대화·근거 검토3건 이상·보완·재확인·최종 반영. PASS 증거 투입 전 점검 항목(배터리/유류, 통신 링크, 적재 중량, 기상)을 합격 기준과 함께 체크리스트로 만들고, 점검 결과를 얼마나 믿을 수 있는지 6단계로 구분한다.의판단이제공자료·역할책임·교육용가정과연결됩니다. 사실/제안/미확인과실제/모의수행을구분합니다. 보완 방법 근거없는원인·규정·완료추정은수정하거나확인필요로보류합니다.

**평가 항목 3:** [진행 추적 및 완료 기준] 실제 조원·두 평가자·2분 판단·20분 문서점검·3분 발표·실제 교관 확인을 AI 모의와 구분한다. 미실시를 PASS로 쓰지 않는다. 파일: framework/log05_source_reliability_matrix.md, LOG05_M02_forms.md, forms/LOG05_M02_forms.md 실제AI요청·답변·근거대조3건이상·보완·재확인·본인최종선택과작성/실행기록이연결됩니다. 최종ZIP생성은실제학습완료·교관승인 인증이아닙니다. 보완 방법 미작성·미실행·검토누락을보완하고교관검토상태를별도로표시합니다.

**중요:** 정답·실제 측정값·사람의 서명이나 승인을 AI가 만들어 주지 않도록 합니다.

In [ ]:
MISSION_JSON = '{"code": "LOG05_m02", "steps": [{"title": "근거 등급과 합격을 분리한다"}, {"title": "AI와 체크리스트·G 기준을 작성한다"}, {"title": "고정 기준으로 독립 검토를 받는다"}, {"title": "최초·수정 기준과 결과를 저장한다"}]}'
# LOG05 학습 기록 도구 (표준 Python/Jupyter 환경용)
import json, zipfile, io, os, datetime
from pathlib import Path
from IPython.display import display, Markdown
try:
    import ipywidgets as widgets
except ImportError as exc:
    raise RuntimeError("ipywidgets가 필요합니다. 폐쇄망에서는 사전 설치된 로컬 Jupyter 또는 제공된 Markdown 양식을 사용하세요.") from exc

MISSION_DATA = json.loads(MISSION_JSON)
CODE = MISSION_DATA['code']
OUT_DIR = Path(CODE + '_results')
OUT_DIR.mkdir(exist_ok=True)
BACKUP = OUT_DIR / (CODE + '_record.json')
fields = {}
for no, item in enumerate(MISSION_DATA['steps'], 1):
    fields['activity_%02d' % no] = widgets.Textarea(
        value='', placeholder='내 판단, 확인한 자료 번호, AI 제안과 수정 이유를 직접 작성하세요.',
        description='활동 %s' % no, layout=widgets.Layout(width='98%',height='130px'),
        style={'description_width':'75px'})
for key, label in [('first_thought','AI에게 묻기 전 내 판단'), ('ai_request','실제로 보낸 AI 요청 전체'),
                   ('ai_response','실제로 받은 AI 답변 전체'), ('source_check','원자료 대조 3건과 수정 이유'),
                   ('final_result','최종 작성 결과와 근거'), ('limits','미실시/미측정/확인 필요 및 교관 검토 상태')]:
    fields[key] = widgets.Textarea(value='', description=label,
       layout=widgets.Layout(width='98%',height='125px'),style={'description_width':'190px'})
status = widgets.HTML(value='<b>아직 저장하지 않았습니다.</b> 작성과 교관 승인은 별개입니다.')
restore_input = widgets.Textarea(description='복원 JSON',placeholder='백업 JSON의 내용을 붙여넣으세요.',layout=widgets.Layout(width='98%',height='100px'),style={'description_width':'110px'})

def payload():
    return {'schema':'log05-student-v1','mission':CODE,'saved_at':datetime.datetime.now(datetime.timezone.utc).isoformat(),
            'answers':{key:w.value for key,w in fields.items()},
            'status':'작성 기록 (학습 PASS 또는 교관 승인 아님)'}

def save_record(_=None):
    obj=payload()
    if BACKUP.exists():
        stamp=datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
        BACKUP.rename(OUT_DIR / (CODE+'_previous_'+stamp+'.json'))
    BACKUP.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding='utf8')
    status.value='<b>로컬 저장 완료:</b> '+str(BACKUP)+' (PASS 아님)'

def export_json(_=None):
    save_record()
    raw=BACKUP.read_bytes()
    try:
        from google.colab import files
        files.download(str(BACKUP))
    except ImportError:
        print('Colab이 아닙니다. 저장 위치:',BACKUP.resolve())

def restore_record(_=None):
    try:
        obj=json.loads(restore_input.value)
        if obj.get('mission')!=CODE or not isinstance(obj.get('answers'),dict):raise ValueError('다른 미션 또는 형식이 맞지 않는 백업입니다.')
        save_record()  # 복원 전 백업 저장
        for key,w in fields.items():w.value=str(obj['answers'].get(key,''))
        status.value='<b>복원 적용:</b> 기존 작성 내용은 이전 JSON으로 보관했습니다. 다시 저장하세요.'
    except Exception as exc:status.value='<b>복원 오류:</b> '+str(exc)

def export_review_zip(_=None):
    save_record()
    target=OUT_DIR / (CODE+'_review.zip')
    with zipfile.ZipFile(target,'w',compression=zipfile.ZIP_DEFLATED) as zz:
        zz.write(BACKUP,arcname=BACKUP.name)
        text='\n\n'.join('## '+key+'\n'+val for key,val in payload()['answers'].items())
        zz.writestr(CODE+'_work.md',text)
        zz.writestr('README.txt','검토용 보관 파일. 실제 평가 PASS 또는 교관 승인 자료가 아닙니다.')
    try:
        from google.colab import files
        files.download(str(target))
    except ImportError:
        print('Colab이 아닙니다. ZIP 위치:',target.resolve())

for i,item in enumerate(MISSION_DATA['steps'],1):
    display(Markdown('### 활동 %d. %s' % (i,item['title'])))
    display(fields['activity_%02d' % i])
for key,label in [('first_thought','최초 판단'),('ai_request','AI 요청 원문'),('ai_response','AI 답변 원문'),
 ('source_check','원자료 대조'),('final_result','최종 결과'),('limits','한계 및 실제 검토 상태')]:
    display(Markdown('### '+label));display(fields[key])
buttons=[]
for title,handler in [('작성 내용 저장',save_record),('작성 JSON 내려받기',export_json),('검토용 ZIP 내려받기',export_review_zip),('작성 JSON 복원',restore_record)]:
    b=widgets.Button(description=title,button_style='info');b.on_click(handler);buttons.append(b)
display(widgets.HBox(buttons[:2]));display(widgets.HBox(buttons[2:]));display(restore_input);display(status)


## 마무리

작성 JSON과 검토용 ZIP을 내려받아 보관하세요. ZIP은 **검토용**이며 평가 PASS나 교관 승인 증명이 아닙니다. 전 단계 결과가 필요하면 본인의 실제 이전 기록을 사용하고, 기록이 없으면 **미실시**로 남기세요.